# Assignment 1: The Headline Test

**AI-assisted analysis of the supplied teaching_prices_clean.csv.**

The code has been executed on the supplied CSV and the actual outputs are saved
below. Analysis run: 2026-11-04 21:48:08 PKT (UTC+05:00).

The file labels every row as illustrative teaching data, not actual PBS
observations. No official data dictionary was supplied; column meanings below
are checked against the header and rows. The separate files contain AI-assisted question and reflection drafts. The student still needs to verify the official dictionary and review the assessed wording under the handout's AI rules.

The student's earlier predictions and attempt were not supplied. They remain
unfilled. This notebook is not evidence of a pre-AI attempt.


## Question drafts

1. How did the recorded price of Staple A in City 1 change between 1 January and 19 March 2026, measured in the dataset's illustrative PKR per kg?
2. For each item on each of the 12 dates, how much did its recorded price in City 2 differ from its price in City 1, using the same illustrative PKR per kg unit?

These are AI-assisted drafts. The corresponding variables, scope and checks are in `1_attempt.md`.

## 1. Preserve your genuine earlier predictions

The actual results are now known. Enter only values from an attempt you really
made before running the analysis. If no prediction was recorded, leave `None`
and explain. Do not reconstruct a prediction from the results below.

In [1]:
predicted_rows = None
predicted_columns = None
predicted_items = None
predicted_dates = None

# Write the actual timing, including timezone. Do not backdate.
prediction_recorded_at = ""
prediction_timing_note = ""  # e.g. before code but after AI, if that is true

## 2. Read the unchanged course CSV

The code searches the included `data` folder from either the project root or
the notebook folder. For the separate notebook download, place the CSV beside
the notebook. The file is read as text to preserve labels. Prices are converted
to numbers only in a separate analysis copy. No source rows or columns are changed.

In [2]:
from pathlib import Path
import pandas as pd

candidates = [
    Path("data/teaching_prices_clean.csv"),
    Path("../../data/teaching_prices_clean.csv"),
    Path("teaching_prices_clean.csv"),
]
path = next((candidate for candidate in candidates if candidate.is_file()), None)
if path is None:
    raise FileNotFoundError("Keep the included data folder, or place the CSV beside this notebook.")

df = pd.read_csv(path, dtype="string", keep_default_na=False)
print("Source file:", path.name)
print("Columns:", df.columns.tolist())
df.head(8)

Source file: teaching_prices_clean.csv
Columns: ['date', 'item', 'city', 'price', 'unit', 'data_label']


,date,item,city,price,unit,data_label
0,2026-01-01,Staple A,City 1,101.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OBSERVATIONS
1,2026-01-01,Staple A,City 2,107.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OBSERVATIONS
2,2026-01-01,Oil B,City 1,298.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OBSERVATIONS
3,2026-01-01,Oil B,City 2,304.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OBSERVATIONS
4,2026-01-01,Pulse C,City 1,200.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OBSERVATIONS
5,2026-01-01,Pulse C,City 2,206.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OBSERVATIONS
6,2026-01-01,Sweetener D,City 1,149.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OBSERVATIONS
7,2026-01-01,Sweetener D,City 2,155.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OBSERVATIONS


## 3. Select the observed item and date columns

`item` and `date` are the actual headers. All observed dates follow YYYY-MM-DD.
There are no empty cells in this file, so no unknown missing-value code has
been removed. The instructor's dictionary remains to be checked.

In [3]:
ITEM_COLUMN = "item"  # Confirmed against the supplied CSV header.
DATE_COLUMN = "date"  # Confirmed against the supplied CSV header.
MISSING_LABELS = [""]  # Add any dictionary-defined missing labels.
DATE_FORMAT = "%Y-%m-%d"  # All observed labels successfully parse in this format.

if not ITEM_COLUMN or not DATE_COLUMN:
    raise ValueError("Enter ITEM_COLUMN and DATE_COLUMN using the data dictionary.")
if ITEM_COLUMN == DATE_COLUMN:
    raise ValueError("The item and date columns must be different.")
for column in [ITEM_COLUMN, DATE_COLUMN]:
    if column not in df.columns:
        raise KeyError(f"Column {column!r} was not found. Check spelling and capitals.")

items = df[ITEM_COLUMN].mask(df[ITEM_COLUMN].isin(MISSING_LABELS))
dates = df[DATE_COLUMN].mask(df[DATE_COLUMN].isin(MISSING_LABELS))

## 4. Check labels and dates

The following outputs check missing values, exact labels, outside spaces and
the observed date format. These checks were run on the supplied file.

In [4]:
print("Missing item values:", int(items.isna().sum()))
print("Missing date values:", int(dates.isna().sum()))
print("Distinct item labels:", sorted(items.dropna().unique().tolist()))
print("Distinct date labels:", sorted(dates.dropna().unique().tolist()))

for name, values in [(ITEM_COLUMN, items), (DATE_COLUMN, dates)]:
    extra_spaces = values.notna() & values.ne(values.str.strip())
    print(f"Values with outside spaces in {name}:", int(extra_spaces.sum()))

if DATE_FORMAT:
    parsed_dates = pd.to_datetime(dates, format=DATE_FORMAT, errors="coerce")
    invalid = dates.notna() & parsed_dates.isna()
    if invalid.any():
        print("Unparsed labels:", dates[invalid].unique().tolist())
        raise ValueError("Investigate the date format or invalid values before continuing.")
    print("Distinct parsed date values:", int(parsed_dates.nunique()))
    if parsed_dates.nunique() != dates.nunique():
        raise ValueError("Different labels map to the same date. Resolve and document this.")
    print("Date parsing check passed using the format you supplied.")
else:
    print("Date parsing is NOT checked. Verify the format and labels before submitting.")

Missing item values: 0
Missing date values: 0
Distinct item labels: ['Oil B', 'Pulse C', 'Staple A', 'Sweetener D']
Distinct date labels: ['2026-01-01', '2026-01-08', '2026-01-15', '2026-01-22', '2026-01-29', '2026-02-05', '2026-02-12', '2026-02-19', '2026-02-26', '2026-03-05', '2026-03-12', '2026-03-19']
Values with outside spaces in item: 0
Values with outside spaces in date: 0
Distinct parsed date values: 12
Date parsing check passed using the format you supplied.


## 5. The three required checks and comparison

Shape is `(rows, columns)`. Repeated item and date labels count once each.
The actual counts can be compared only with genuine earlier predictions.
`Not supplied` does not mean the student never made an attempt; no attempt
or predictions were provided in this conversation.

In [5]:
actual_rows, actual_columns = df.shape
actual_items = int(items.nunique(dropna=True))
actual_dates = int(dates.nunique(dropna=True))

print("Shape (rows, columns):", df.shape)
print("Distinct items:", actual_items)
print("Distinct dates (as recorded):", actual_dates)

checks = [
    ("Rows", predicted_rows, actual_rows),
    ("Columns", predicted_columns, actual_columns),
    ("Distinct items", predicted_items, actual_items),
    ("Distinct dates", predicted_dates, actual_dates),
]
comparison_rows = []
for label, prediction, actual in checks:
    if prediction is None:
        comparison_rows.append([label, "Not supplied", actual, "Not available"])
    else:
        if not isinstance(prediction, int) or isinstance(prediction, bool) or prediction < 0:
            raise ValueError(f"{label}: use a non-negative whole-number prediction or None.")
        comparison_rows.append([label, prediction, actual, actual - prediction])

comparison = pd.DataFrame(
    comparison_rows,
    columns=["Check", "My original prediction", "Actual result", "Actual minus prediction"],
)
print("Prediction timing:", prediction_recorded_at or "Not supplied")
print("Timing note:", prediction_timing_note or "Not supplied")
comparison

Shape (rows, columns): (96, 6)
Distinct items: 4
Distinct dates (as recorded): 12
Prediction timing: Not supplied
Timing note: Not supplied


,Check,My original prediction,Actual result,Actual minus prediction
0,Rows,Not supplied,96,Not available
1,Columns,Not supplied,6,Not available
2,Distinct items,Not supplied,4,Not available
3,Distinct dates,Not supplied,12,Not available


## 6. Explanation of the actual counts

- **Shape: (96, 6).** The file has 96 data rows and the six fields `date`, `item`,
  `city`, `price`, `unit` and `data_label`. The header is not a data row.
- **Distinct items: 4.** Staple A, Oil B, Pulse C and Sweetener D.
- **Distinct dates: 12.** From 1 January to 19 March 2026, seven days apart.
- There are four items in two cities on every date: **4 x 2 x 12 = 96**.
- Each record has the unit `PKR per kg (illustrative)`.
- No empty fields or duplicate `(date, item, city)` records were found.

**Prediction comparison:** No original predictions were supplied, so no prediction difference is claimed.

**Dictionary check:** Not completed because the official dictionary was not supplied. The observed header and values have been checked.

The following optional checks support question selection. The required numerical
task above is shape, distinct items and distinct dates.

In [6]:
empty_cells = int(df.eq("").sum().sum())
print("Empty cells:", empty_cells)
print("Duplicate whole rows:", int(df.duplicated().sum()))
print("Duplicate item-city-date keys:", int(df.duplicated(["date", "item", "city"]).sum()))
print("Records per date:")
print(df.groupby("date").size().to_string())
print("Data label:", df["data_label"].unique().tolist())

Empty cells: 0
Duplicate whole rows: 0
Duplicate item-city-date keys: 0
Records per date:
date
2026-01-01    8
2026-01-08    8
2026-01-15    8
2026-01-22    8
2026-01-29    8
2026-02-05    8
2026-02-12    8
2026-02-19    8
2026-02-26    8
2026-03-05    8
2026-03-12    8
2026-03-19    8
Data label: ['ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OBSERVATIONS']


## 7. Optional evidence for the proposed questions

Compare the same item, city and unit at the first and last dates. Percentage
change uses the first price as its denominator. These are illustrative prices.

In [7]:
prices = df.copy()
prices["price"] = pd.to_numeric(prices["price"], errors="raise")
prices["date"] = pd.to_datetime(prices["date"], format=DATE_FORMAT, errors="raise")
first_date = prices["date"].min()
last_date = prices["date"].max()
keys = ["item", "city", "unit"]
first = prices.loc[prices["date"].eq(first_date), keys + ["price"]]
last = prices.loc[prices["date"].eq(last_date), keys + ["price"]]
endpoint_prices = first.merge(last, on=keys, suffixes=("_first", "_last"), validate="one_to_one")
endpoint_prices["change_PKR"] = endpoint_prices["price_last"] - endpoint_prices["price_first"]
endpoint_prices["change_percent"] = endpoint_prices["change_PKR"] / endpoint_prices["price_first"] * 100
print("First date:", first_date.date(), "Last date:", last_date.date())
print("Unit:", df["unit"].unique().tolist())
endpoint_prices.drop(columns="unit").round(2)

First date: 2026-01-01 Last date: 2026-03-19
Unit: ['PKR per kg (illustrative)']


,item,city,price_first,price_last,change_PKR,change_percent
0,Staple A,City 1,101.0,114.5,13.5,13.37
1,Staple A,City 2,107.0,120.5,13.5,12.62
2,Oil B,City 1,298.0,344.0,46.0,15.44
3,Oil B,City 2,304.0,350.0,46.0,15.13
4,Pulse C,City 1,200.0,207.5,7.5,3.75
5,Pulse C,City 2,206.0,213.5,7.5,3.64
6,Sweetener D,City 1,149.0,178.5,29.5,19.8
7,Sweetener D,City 2,155.0,184.5,29.5,19.03


In [8]:
matched = prices.pivot(index=["date", "item", "unit"], columns="city", values="price")
city_gap = matched["City 2"] - matched["City 1"]
print("Matched item-date comparisons:", len(city_gap))
print("Incomplete city pairs:", int(matched[["City 1", "City 2"]].isna().any(axis=1).sum()))
print("Distinct City 2 minus City 1 gaps (illustrative PKR/kg):", sorted(city_gap.unique().tolist()))

Matched item-date comparisons: 48
Incomplete city pairs: 0
Distinct City 2 minus City 1 gaps (illustrative PKR/kg): [6.0]


## 8. What the evidence supports

- Staple A in City 1 changes from 101.0 to 114.5: +13.5 PKR/kg, or about 13.37%.
- Sweetener D has the largest percentage rise in City 1 among the four items:
  about 19.80%. Oil B has the largest absolute rise: 46.0 PKR/kg.
- City 2 is 6.0 PKR/kg higher in all 48 matched item-date comparisons.
- There are no income, basket-quantity or purchase-quantity fields. The file
  cannot answer the proposed affordability and causal buying-response questions.
- These are teaching observations, not evidence of actual PBS prices or national inflation.

The official dictionary check remains. Two AI-assisted question drafts are included in 1_attempt.md for student review.
See `3_log_reflection.md` for all five candidate questions and their evidence.

## Official software references

Open and record any source you rely on in your AI log.

- pandas shape: https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.shape.html
- pandas distinct count: https://pandas.pydata.org/docs/reference/api/pandas.Series.nunique.html
- pandas date parsing: https://pandas.pydata.org/docs/reference/api/pandas.to_datetime.html
- VS Code notebooks: https://code.visualstudio.com/docs/datascience/jupyter-notebooks

These references explain the code. They are not sources for claims about prices.